[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcivardi/divelab/blob/editorial-v2/notebooks/18_Integrated_DiveLab_Capstone.ipynb)

# Notebook 18 — Integrated DiveLab Capstone

**Companion to Chapter 18**

This capstone combines two experiments: a synthetic depth history propagated through the principal DiveLab subsystems, and a compact local feedback loop in which the vertical plant generates depth. It introduces no new operational algorithm: its purpose is to reveal coupling, state, uncertainty, control, and different time scales.

> **Educational scope.** The profile and calculations are synthetic. This notebook is not a dive planner, decompression tool, gas-planning tool, training standard, or substitute for a validated dive computer.

## Learning objectives

- connect pressure, gas volume, buoyancy, vertical motion, feedback control, gas demand, sensor estimation, and inert-gas states;
- distinguish common input history from subsystem state;
- compare prescribed-profile propagation with a local closed-loop simulation;
- identify assumptions and validation boundaries in an integrated model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

plt.rcParams.update({"figure.figsize":(9,5),"axes.grid":True})
rho,g,p0=1025.0,9.80665,101325.0

def ambient_pressure_pa(depth_m):
    return p0+rho*g*np.asarray(depth_m)

def ambient_pressure_bar(depth_m):
    return ambient_pressure_pa(depth_m)/1e5

## 1. One synthetic depth history

The profile is designed only to exercise the models: descent, a changing-depth interval, ascent, and a long surface observation. Cylinder-gas withdrawal ends when the synthetic dive ends; the surface interval remains in the timeline so the inert-gas states can continue evolving.

In [ ]:
def synthetic_depth(t_min):
    t=np.asarray(t_min)
    z=np.zeros_like(t,dtype=float)
    z=np.where(t<3,18*t/3,z)
    z=np.where((t>=3)&(t<15),18,z)
    z=np.where((t>=15)&(t<20),18-6*(t-15)/5,z)
    z=np.where((t>=20)&(t<25),12,z)
    z=np.where((t>=25)&(t<29),12*(29-t)/4,z)
    return np.maximum(z,0)

t=np.linspace(0,180,18001)
z=synthetic_depth(t)
p_bar=ambient_pressure_bar(z)

fig,ax=plt.subplots(2,1,sharex=True,figsize=(9,6))
ax[0].plot(t,z); ax[0].invert_yaxis(); ax[0].set_ylabel("Depth [m]")
ax[1].plot(t,p_bar); ax[1].set(xlabel="Time [min]",ylabel="Pressure [bar abs]")
plt.show()

## 2. Flexible gas volume and buoyancy contribution

A reference gas volume follows Boyle's law. Tidal lung-volume variation is added separately as a periodic buoyancy disturbance while submerged.

In [ ]:
v_surface=8e-3
v_gas=v_surface*p0/ambient_pressure_pa(z)
submerged=z>0.1
tidal=0.35e-3*np.sin(2*np.pi*0.25*60*t)*submerged
tidal_force=rho*g*tidal

fig,ax=plt.subplots(2,1,sharex=True,figsize=(9,6))
ax[0].plot(t,1e3*v_gas); ax[0].set_ylabel("Flexible gas [L]")
ax[1].plot(t,tidal_force); ax[1].set(xlabel="Time [min]",ylabel="Tidal buoyancy [N]")
ax[1].set_xlim(0,8)
plt.show()

## 3. A noisy pressure sensor and derived depth

Pressure is measured; depth is derived using estimated environmental parameters.

In [ ]:
rng=np.random.default_rng(18)
sample_t=np.arange(0,180.01,1/60)  # one sample per second
z_sample=synthetic_depth(sample_t)
p_measured=ambient_pressure_pa(z_sample)+300+rng.normal(0,220,len(sample_t))
z_derived=(p_measured-p0)/(rho*g)

def lowpass(x,alpha=0.10):
    y=np.empty_like(x); y[0]=x[0]
    for k in range(1,len(x)): y[k]=alpha*x[k]+(1-alpha)*y[k-1]
    return y

z_est=lowpass(z_derived)
descent_rate=np.gradient(z_est,sample_t*60)
fig,ax=plt.subplots(2,1,sharex=True,figsize=(9,6))
ax[0].plot(sample_t,z_sample,label="true"); ax[0].plot(sample_t,z_est,label="estimated")
ax[0].invert_yaxis(); ax[0].set_ylabel("Depth [m]"); ax[0].legend()
ax[1].plot(sample_t,descent_rate); ax[1].set(xlabel="Time [min]",ylabel="Downward speed [m/s]")
ax[1].set_xlim(0,35); plt.show()

## 4. Remaining gas as a resource state

Surface-equivalent respiratory demand is pressure-scaled and integrated. The assumed workload is deliberately visible.

In [ ]:
surface_rate=18.0
workload=1+0.45*((sample_t>=10)&(sample_t<14))
using_cylinder=sample_t<29.0
gas_rate=surface_rate*workload*ambient_pressure_pa(np.maximum(z_est,0))/p0*using_cylinder
dt_min=np.diff(sample_t)
used=np.concatenate([[0],np.cumsum((gas_rate[1:]+gas_rate[:-1])*dt_min/2)])
remaining=3000-used

fig,ax=plt.subplots(2,1,sharex=True,figsize=(9,6))
ax[0].plot(sample_t,gas_rate); ax[0].set_ylabel("Demand [surface L/min]")
ax[1].plot(sample_t,remaining); ax[1].set(xlabel="Time [min]",ylabel="Modelled gas [surface L]")
ax[1].set_xlim(0,40); plt.show()
assert np.all(np.diff(remaining)<=1e-10)
assert np.min(remaining)>=0

## 5. Hidden inert-gas states

The same estimated pressure history drives a bank of educational nitrogen compartments. No ceiling or schedule is calculated.

In [ ]:
f_n2,p_h2o=0.79,0.0627
half_times=np.array([5.,20.,80.,240.])
inspired=f_n2*(ambient_pressure_bar(np.maximum(z_est,0))-p_h2o)
tissue=np.empty((len(half_times),len(sample_t)))
tissue[:,0]=f_n2*(ambient_pressure_bar(0)-p_h2o)
k=np.log(2)/half_times
for j,dt in enumerate(np.diff(sample_t)):
    tissue[:,j+1]=inspired[j]+(tissue[:,j]-inspired[j])*np.exp(-k*dt)

for i,h in enumerate(half_times): plt.plot(sample_t,tissue[i],label=f"{h:g} min")
plt.plot(sample_t,inspired,"k--",alpha=.5,label="inspired N$_2$")
plt.xlabel("Time [min]"); plt.ylabel("Nitrogen pressure [bar]")
plt.title("Different states remember different portions of history"); plt.legend(); plt.show()

## 6. Integrated system dashboard

In [ ]:
fig,ax=plt.subplots(4,1,sharex=True,figsize=(10,10))
ax[0].plot(sample_t,z_est); ax[0].invert_yaxis(); ax[0].set_ylabel("Depth [m]")
ax[1].plot(sample_t,ambient_pressure_bar(np.maximum(z_est,0))); ax[1].set_ylabel("Pressure [bar]")
ax[2].plot(sample_t,remaining); ax[2].set_ylabel("Gas [surface L]")
for i,h in enumerate(half_times): ax[3].plot(sample_t,tissue[i],label=f"{h:g} min")
ax[3].set(xlabel="Time [min]",ylabel="N$_2$ [bar]"); ax[3].legend(ncol=4)
fig.suptitle("One depth history, several coupled subsystem responses")
plt.tight_layout(); plt.show()

## 7. Close the local vertical-motion loop

The dashboard above prescribes depth so that subsystem propagation can be studied independently. We now use the local plant from Part II and the PD state feedback from Part III so that depth is generated by the model:

$$\delta\dot z=-\delta v,\qquad \delta\dot v=a\delta z+\frac{u+d}{m},\qquad u=K_p\delta z-K_d\delta v.$$

Depth is positive downward, velocity and force are positive upward, and $d$ is an unmodelled upward force. The force limit represents generic actuator authority; it is not a command for a diver or a particular device. This local experiment assumes the state is available to the controller. Chapters 7 and 8 explain why a practical implementation would instead use measured or estimated signals.

In [ ]:
mass=85.0                         # kg
reference_depth=18.0             # m
reference_pressure=ambient_pressure_pa(reference_depth)
reference_gas=v_surface*p0/reference_pressure
buoyancy_slope=-rho**2*g**2*reference_gas/reference_pressure
plant_a=buoyancy_slope/mass
desired_poles=np.array([-0.35,-0.55])
K_d=-mass*np.sum(desired_poles)
K_p=mass*(np.prod(desired_poles)-plant_a)
force_limit=12.0                  # N, symmetric educational limit

def local_closed_loop_rhs(time_s,state):
    depth_error,upward_velocity=state
    force_raw=K_p*depth_error-K_d*upward_velocity
    force_applied=np.clip(force_raw,-force_limit,force_limit)
    disturbance=5.0 if time_s>=12.0 else 0.0
    return [-upward_velocity,plant_a*depth_error+(force_applied+disturbance)/mass]

closed_loop_poles=np.roots([1,K_d/mass,plant_a+K_p/mass])
time_s=np.linspace(0,60,3001)
closed_loop=solve_ivp(local_closed_loop_rhs,(0,60),[0.80,0.0],t_eval=time_s,
                            rtol=1e-9,atol=1e-11)
depth_error,upward_velocity=closed_loop.y
closed_loop_depth=reference_depth+depth_error
force_raw=K_p*depth_error-K_d*upward_velocity
force_applied=np.clip(force_raw,-force_limit,force_limit)
pressure_change=(ambient_pressure_pa(closed_loop_depth)-reference_pressure)/1e3
closed_loop_gas=v_surface*p0/ambient_pressure_pa(closed_loop_depth)
buoyancy_change=rho*g*(closed_loop_gas-reference_gas)

print(f"Closed-loop poles: {closed_loop_poles[0]:.3f}, {closed_loop_poles[1]:.3f} s^-1")
print(f"Final depth error: {depth_error[-1]:.3f} m")
assert np.all(closed_loop_poles.real<0)
assert np.max(np.abs(force_applied))<=force_limit+1e-12

fig,ax=plt.subplots(5,1,sharex=True,figsize=(9,11))
ax[0].plot(time_s,closed_loop_depth); ax[0].invert_yaxis(); ax[0].set_ylabel("Depth [m]")
ax[1].plot(time_s,upward_velocity); ax[1].set_ylabel("Upward speed [m/s]")
ax[2].plot(time_s,force_applied,label="applied force")
ax[2].axhline(force_limit,color="k",ls=":"); ax[2].axhline(-force_limit,color="k",ls=":")
ax[2].set_ylabel("Force [N]"); ax[2].legend()
ax[3].plot(time_s,pressure_change); ax[3].set_ylabel("Pressure change [kPa]")
ax[4].plot(time_s,buoyancy_change); ax[4].set(xlabel="Time [s]",ylabel="Buoyancy change [N]")
for axis in ax: axis.axvline(12,color="tab:red",ls="--",alpha=.6)
fig.suptitle("Local feedback closes the motion–pressure–buoyancy loop")
plt.tight_layout(); plt.show()

The stable poles predict recovery from the initial depth error. When the persistent disturbance begins at $12~\mathrm{s}$, PD feedback bounds the response but leaves a small offset, as Chapter 10 predicts. The resulting depth changes pressure and flexible-gas buoyancy, completing the local physical loop. Saturation and the exact Boyle-law output also show where the linear pole calculation stops describing the whole experiment.

## 8. What the capstone does—and does not—mean

- Pressure follows depth almost immediately.
- Flexible gas volume changes algebraically with pressure.
- Remaining gas carries the integral of past demand.
- Compartment states retain pressure history at several time scales.
- Sensor error propagates into every model driven by estimated depth.

The dashboard and closed-loop experiment do **not** demonstrate that the synthetic profile is safe, that the remaining gas is adequate, that the controller is suitable for real equipment, or that a real diver would follow these predictions.

## Capstone exercises

1. Classify every variable as state, input, disturbance, parameter, measurement, or derived output.
2. Add uncertainty bands for breathing demand and sensor bias.
3. Compare two depth histories with the same maximum depth and duration.
4. Change the disturbance or force limit in the closed-loop experiment and explain which linear predictions remain valid.
5. Write a validation matrix stating what evidence each subsystem would require.


In [ ]:
# Capstone starter: compare two histories with equal maximum depth
# Define a second synthetic_depth() function, then compare gas and tissue states.


## Final reflection

You began with depth and pressure. You can now trace one depth change through gas compression, buoyancy, motion, sensing, estimation, control, resource depletion, and inert-gas state. The durable skill is not memorizing the dashboard; it is knowing what is connected, what is hidden, what is assumed, and what still requires evidence.